In [ ]:
# FHIR Silver + Gold Layers
# Cleans and standardizes Bronze, deduplicates incoming records,
# maintains SCD Type 2 history, and creates Gold analytical tables.


StatementMeta(, aa3dd163-371f-460d-a118-3979d3f3c4bf, 3, Finished, Available, Finished, False)

In [ ]:
extraction_date = "2026-10-14"
resources = "Patient,Encounter,Observation,Condition"

StatementMeta(, aa3dd163-371f-460d-a118-3979d3f3c4bf, 4, Finished, Available, Finished, False)

In [ ]:
RESOURCES = [r.strip() for r in resources.split(",") if r.strip()]

StatementMeta(, aa3dd163-371f-460d-a118-3979d3f3c4bf, 5, Finished, Available, Finished, False)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# RESOURCES = ["Patient", "Encounter", "Observation", "Condition"]

def flatten_structs(df):
    # Flatten nested structs recursively; preserve arrays as JSON strings.
    while True:
        struct_fields = [
            field.name for field in df.schema.fields
            if field.dataType.typeName() == "struct"
        ]
        if not struct_fields:
            break

        columns = []
        for field in df.schema.fields:
            name = field.name
            dtype = field.dataType

            if dtype.typeName() == "struct":
                for nested_field in dtype.fields:
                    columns.append(
                        F.col(f"`{name}`.`{nested_field.name}`")
                         .alias(f"{name}_{nested_field.name}")
                    )
            elif dtype.typeName() == "array":
                columns.append(F.to_json(F.col(f"`{name}`")).alias(name))
            else:
                columns.append(F.col(f"`{name}`"))

        df = df.select(*columns)

    # Convert any remaining arrays after recursive flattening.
    for field in df.schema.fields:
        if field.dataType.typeName() == "array":
            df = df.withColumn(
                field.name,
                F.to_json(F.col(f"`{field.name}`"))
            )
    return df

def add_scd2_metadata(df):
    metadata_cols = {
        "extraction_timestamp",
        "bronze_loaded_timestamp",
        "silver_processed_timestamp",
        "valid_from",
        "valid_to",
        "is_current",
        "record_hash"
    }
    business_columns = [c for c in df.columns if c not in metadata_cols]
    hash_inputs = [
        F.coalesce(F.col(c).cast("string"), F.lit("NULL"))
        for c in business_columns
    ]

    return (
        df.withColumn(
            "record_hash",
            F.sha2(F.concat_ws("||", *hash_inputs), 256)
        )
        .withColumn("valid_from", F.current_timestamp())
        .withColumn("valid_to", F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
        .withColumn("silver_processed_timestamp", F.current_timestamp())
    )

def upsert_scd2(source_df, target_table, key_column="id"):
    # First run creates the full SCD2 table.
    if not spark.catalog.tableExists(target_table):
        (source_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table))
        print(f"Created {target_table}")
        return

    target = DeltaTable.forName(spark, target_table)
    current_df = (
        spark.table(target_table)
             .filter(F.col("is_current") == True)
             .select(key_column, "record_hash")
    )

    changes = (
        source_df.alias("s")
        .join(
            current_df.alias("t"),
            F.col(f"s.{key_column}") == F.col(f"t.{key_column}"),
            "left"
        )
        .filter(
            F.col(f"t.{key_column}").isNull() |
            (F.col("s.record_hash") != F.col("t.record_hash"))
        )
        .select("s.*")
    )

    if changes.limit(1).count() == 0:
        print(f"No SCD2 changes for {target_table}")
        return

    # Expire existing current versions for changed keys.
    target.alias("t").merge(
        changes.alias("s"),
        f"t.{key_column} = s.{key_column} "
        f"AND t.is_current = true "
        f"AND t.record_hash <> s.record_hash"
    ).whenMatchedUpdate(
        set={
            "valid_to": "current_timestamp()",
            "is_current": "false"
        }
    ).execute()

    # Append the new versions. New keys are included here too.
    (changes.write
        .format("delta")
        .mode("append")
        .saveAsTable(target_table))

    print(f"Applied {changes.count()} SCD2 change(s) to {target_table}")

for resource in RESOURCES:
    source_table = f"bronze_{resource.lower()}"
    target_table = f"silver_{resource.lower()}"

    print(f"Processing {source_table} -> {target_table}")

    silver_df = flatten_structs(spark.table(source_table))

    # Standardize whitespace in all string columns.
    for field in silver_df.schema.fields:
        if field.dataType.typeName() == "string":
            silver_df = silver_df.withColumn(
                field.name,
                F.when(
                    F.trim(F.col(f"`{field.name}`")) == "",
                    F.lit(None)
                ).otherwise(F.trim(F.col(f"`{field.name}`")))
            )

    # Every core FHIR resource must have an id.
    silver_df = silver_df.filter(F.col("id").isNotNull())

    # Deduplicate the incoming Bronze records by latest load.
    w = Window.partitionBy("id").orderBy(
        F.col("bronze_loaded_timestamp").desc_nulls_last()
    )
    silver_df = (
        silver_df
        .withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    silver_df = add_scd2_metadata(silver_df)
    upsert_scd2(silver_df, target_table)


StatementMeta(, aa3dd163-371f-460d-a118-3979d3f3c4bf, 6, Finished, Available, Finished, False)

Processing bronze_patient -> silver_patient
No SCD2 changes for silver_patient
Processing bronze_encounter -> silver_encounter
No SCD2 changes for silver_encounter
Processing bronze_observation -> silver_observation
No SCD2 changes for silver_observation
Processing bronze_condition -> silver_condition
No SCD2 changes for silver_condition
